# Lab 73: The Cheater Detector

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-73.ipynb)

**What you will do:** test whether a logic puzzle about catching cheaters is easier than the same puzzle stated abstractly, compare your results with the book's typical figures, check an AI chatbot for the same split, and see how a fairness-driven responder's earnings compare with a purely rational one in the related Ultimatum Game simulation.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 73 you gave a friend two versions of the same logic puzzle: an abstract version
(letters and numbers, as in Lab 60) and a socially framed version about checking whether
employees who received a bonus worked overtime. Both versions have exactly the same logical
structure, but most people solve the social version far more easily. This gap is the signature
of the **cheater-detection hypothesis** (Cosmides & Tooby, 1992): the brain reasons about rule
violations more effectively when the rule is a social contract than when it is abstract.

## Record your results

Send the abstract version first, then the social version, to a handful of friends (or answer
both yourself). Log whether each person got each version right.

In [ ]:
# example data: replace with yours -- record whether each person answered correctly
wason_df = pd.DataFrame({
    "person": ["You", "Friend 1", "Friend 2", "Friend 3", "Friend 4"],
    "abstract_correct": [False, False, True, False, False],   # letters-and-numbers version
    "social_correct": [True, True, True, False, True],        # employees-and-bonuses version
})
display(wason_df)
pct_abstract = wason_df.abstract_correct.mean() * 100
pct_social = wason_df.social_correct.mean() * 100
pd.Series({"Abstract version": pct_abstract, "Social version": pct_social}).plot.bar(
    color=["grey", "steelblue"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.xticks(rotation=0); plt.show()
print(f"Abstract version: {pct_abstract:.0f}% correct")
print(f"Social version: {pct_social:.0f}% correct")

## Compare

Lab 73 reports that, typically, fewer than 20% of people get the abstract version right,
while 70-80% get the social version right. Check your own small sample against that pattern.

In [ ]:
print(f"Your group -- abstract version: {pct_abstract:.0f}% correct")
print(f"Your group -- social version: {pct_social:.0f}% correct")
print("Lab 73: typically fewer than 20% correct on the abstract version, 70-80% on the social version.")
if pct_abstract < 20 and 70 <= pct_social <= 80:
    print("Your group matches the typical pattern closely.")
elif pct_social > pct_abstract:
    print("Your group shows the expected direction (social framing easier), even if the exact")
    print("percentages differ -- a handful of people is a very small, noisy sample.")
else:
    print("Your group does not show the usual social-framing advantage. With only a few")
    print("people this could easily be chance; pool more responses in the class data section below.")

## Simulation

Lab 73 showed that people solve the same logical puzzle far more accurately when it is framed as catching a rule-breaker than when it is presented abstractly. The Ultimatum Game shows a related sensitivity to unfairness (Güth, Schmittberger, & Schwarze, 1982). One player proposes how to split a sum of money; the other can accept the split or reject it, in which case neither player gets anything. People routinely reject offers they judge unfair, even though rejecting costs them money—something pure self-interest cannot explain. This simulation runs 10,000 rounds of the Ultimatum Game and compares a "rational" player (accepts any offer above zero) with a "fairness" player (rejects offers below 30%).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_rounds = 10000
pot = 100

# Proposer: offers between 1% and 50% of the pot
offers = np.random.randint(1, 51, n_rounds)

# Rational responder: accepts anything > 0
rational_accept = offers > 0
rational_earnings = np.where(rational_accept, offers, 0)

# Fairness responder: rejects offers below 30%
fairness_threshold = 30
fairness_accept = offers >= fairness_threshold
fairness_earnings = np.where(fairness_accept, offers, 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(offers, bins=range(0, 52, 2), color="steelblue", alpha=0.7,
    edgecolor="white")
ax1.axvline(fairness_threshold, color="red", linestyle="--",
    linewidth=2, label=f"Fairness threshold ({fairness_threshold}%)")
ax1.set_xlabel("Offer (%)")
ax1.set_ylabel("Frequency")
ax1.set_title("Distribution of Offers")
ax1.legend()

labels = ["Rational\n(accept all)", "Fairness\n(reject < 30%)"]
means = [np.mean(rational_earnings), np.mean(fairness_earnings)]
accept_rates = [np.mean(rational_accept) * 100,
    np.mean(fairness_accept) * 100]

bars = ax2.bar(labels, means, color=["grey", "steelblue"], alpha=0.7)
for bar, rate in zip(bars, accept_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"Accept: {rate:.0f}%", ha="center", fontsize=9)
ax2.set_ylabel("Mean earnings per round")
ax2.set_title("Earnings: Rationality vs. Fairness")

plt.tight_layout()
plt.show()

print(f"Rational player: mean earnings = {np.mean(rational_earnings):.1f}, "
      f"accepts {np.mean(rational_accept) * 100:.0f}% of offers")
print(f"Fairness player: mean earnings = {np.mean(fairness_earnings):.1f}, "
      f"accepts {np.mean(fairness_accept) * 100:.0f}% of offers")

**What to notice:** The code prints the rational player accepting 100% of offers for a mean of about 25 points per round, against the fairness player accepting only around 42% of offers (everything at or above the 30% threshold) for a lower mean of about 17 points per round. In a single round, rejecting unfair offers is a net loss—the fairness player leaves money on the table. This simulation only plays one round per pairing, so it cannot show the next step in the logic, but the wider experimental literature on repeated ultimatum games finds it: when proposers face responders who reliably reject low offers, they learn to offer more equitable splits over repeated rounds. Costly punishment is individually irrational in the single round shown here, but socially useful over time—it stabilises cooperation by making cheating unprofitable.

## Change the parameters

**fairness_threshold** is the minimum percentage of the pot the fairness player will
accept. **max_offer** caps how generous a proposer can ever be. **n_rounds** controls how
many simulated rounds are averaged.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def explore(fairness_threshold=30, max_offer=50, n_rounds=10000):
    np.random.seed(42)
    offers = np.random.randint(1, max_offer + 1, n_rounds)

    rational_accept = offers > 0
    rational_earnings = np.where(rational_accept, offers, 0)

    fairness_accept = offers >= fairness_threshold
    fairness_earnings = np.where(fairness_accept, offers, 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(offers, bins=range(0, max_offer + 2, 2), color="steelblue", alpha=0.7, edgecolor="white")
    ax1.axvline(fairness_threshold, color="red", linestyle="--", linewidth=2, label=f"Fairness threshold ({fairness_threshold}%)")
    ax1.set_xlabel("Offer (%)")
    ax1.set_ylabel("Frequency")
    ax1.set_title("Distribution of Offers")
    ax1.legend()

    labels = ["Rational\n(accept all)", "Fairness\n(reject below threshold)"]
    means = [np.mean(rational_earnings), np.mean(fairness_earnings)]
    accept_rates = [np.mean(rational_accept) * 100, np.mean(fairness_accept) * 100]
    bars = ax2.bar(labels, means, color=["grey", "steelblue"], alpha=0.7)
    for bar, rate in zip(bars, accept_rates):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"Accept: {rate:.0f}%", ha="center", fontsize=9)
    ax2.set_ylabel("Mean earnings per round")
    ax2.set_title("Earnings: Rationality vs. Fairness")
    plt.tight_layout()
    plt.show()

    print(f"Rational player: mean earnings = {np.mean(rational_earnings):.1f}, accepts {np.mean(rational_accept) * 100:.0f}% of offers")
    print(f"Fairness player: mean earnings = {np.mean(fairness_earnings):.1f}, accepts {np.mean(fairness_accept) * 100:.0f}% of offers")

interact(explore,
         fairness_threshold=IntSlider(value=30, min=5, max=50, step=5),
         max_offer=IntSlider(value=50, min=20, max=50, step=5),
         n_rounds=IntSlider(value=10000, min=1000, max=20000, step=1000));

## The AI side

The book's "AI extension" suggests testing a chatbot on both framings for consistency. Below,
a small open instruction-tuned model answers the abstract and social versions of the same
puzzle, so you can check whether it treats them the same way -- unlike the split seen in
human performance above.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline
solver = pipeline("text2text-generation", model="google/flan-t5-small")

abstract_prompt = (
    "Cards show A, K, 4, 7, one side visible. Rule: if a card has a vowel on one side, then "
    "it has an even number on the other side. Which cards must you turn over to check the "
    "rule? Answer with card labels only."
)
social_prompt = (
    "Company rule: if an employee got a bonus, they must have worked overtime. Employee A "
    "got a bonus. Employee B did not get a bonus. Employee C worked overtime. Employee D did "
    "not work overtime. Which employees' records must you check? Answer with employee "
    "labels only."
)
abstract_answer = solver(abstract_prompt, max_new_tokens=20)[0]["generated_text"]
social_answer = solver(social_prompt, max_new_tokens=20)[0]["generated_text"]
print("Abstract version -- model answered:", abstract_answer, "(correct: A and 7)")
print("Social version   -- model answered:", social_answer, "(correct: A and D)")

abstract_ok = "A" in abstract_answer and "7" in abstract_answer
social_ok = "A" in social_answer and "D" in social_answer
print(f"\nModel correct on abstract version: {abstract_ok}")
print(f"Model correct on social version: {social_ok}")
print("Consistent" if abstract_ok == social_ok else "Split, like the human pattern above")

flan-t5-small is far smaller and weaker than the large chatbots (GPT-4, Claude, and similar)
the book has in mind, so treat its answers as an illustration of the method, not a strong
replication. The deeper point still holds either way: whether a modern AI system gets both
versions right or both wrong, it is applying one uniform reasoning process to both prompts.
It has no separate "social contract" circuit to switch on for the employees-and-bonuses
version and off for the letters-and-numbers version, which is exactly the split the human
data above is expected to show.

## Pool the class data

Pool percent correct across the class for both versions to see the social-contract advantage
on a larger, less noisy sample.

In [ ]:
import io
csv_text = """id,abstract_correct,social_correct
P01,0,1
P02,0,1
P03,1,1
P04,0,0
P05,0,1
P06,0,1
P07,1,1
P08,0,0
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
diff = class_df.social_correct - class_df.abstract_correct
boot = [diff.sample(len(diff), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class: {class_df.abstract_correct.mean() * 100:.0f}% correct on the abstract version,")
print(f"       {class_df.social_correct.mean() * 100:.0f}% correct on the social version.")
print(f"Mean advantage of the social framing: {diff.mean() * 100:.0f} percentage points",
      f"(95% bootstrap interval {lo * 100:.0f} to {hi * 100:.0f})")
class_df[["abstract_correct", "social_correct"]].mean().mul(100).plot.bar(
    color=["grey", "steelblue"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.xticks(rotation=0); plt.show()

## Questions to think about

1. Lower fairness_threshold to 5. Does the fairness player's mean earnings now approach the rational player's? Roughly what threshold value makes the two strategies earn about the same in this single-round game?
2. Raise fairness_threshold to 50, so only an exact even split is accepted. What happens to the fairness player's earnings and acceptance rate, and why might being "too" fair start to cost more than it protects?
3. The simulation's "What to notice" text says a single round cannot show proposers learning to offer more when they face fairness-minded responders, even though the wider literature on repeated ultimatum games finds exactly that. What would you need to add to this code, given that "offers" is currently just a fixed random array, to let a proposer's behaviour adapt across rounds?
4. Reduce max_offer to 30, so proposers never offer more than 30 percent of the pot. Does the cost of being a fairness player change? What does that reveal about how proposer behaviour and responder strategy depend on each other?

## Challenge

This single-round simulation cannot show proposers learning from rejections. Extend it with
a simple multi-round loop where a proposer's average offer drifts upward after a rejection
and downward after an acceptance, to see whether costly punishment by a fairness responder
can make generosity the more profitable long-run strategy for the proposer.

In [ ]:
# A simple multi-round extension: the proposer nudges its average offer up after a
# rejection and down after an acceptance. This is a toy learning rule for illustration,
# not a replication of a specific published model.
np.random.seed(1)
n_generations = 30
offer_mean = 15.0   # the proposer starts out stingy
fairness_threshold = 30
history = []

for gen in range(n_generations):
    offer = np.clip(np.random.normal(offer_mean, 5), 1, 50)
    accepted = offer >= fairness_threshold
    history.append(offer)
    if not accepted:
        offer_mean += 1.0   # rejection -> offer more next time
    else:
        offer_mean -= 0.1   # acceptance -> can afford to offer slightly less

plt.figure(figsize=(6, 3.5))
plt.plot(history, "o-", color="steelblue")
plt.axhline(fairness_threshold, color="red", linestyle="--", label="Fairness threshold")
plt.xlabel("Round")
plt.ylabel("Offer (%)")
plt.legend()
plt.title("Does rejecting unfair offers teach the proposer to be fairer?")
plt.show()
print("This toy learning rule is illustrative, not a replication of any specific published model.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-73.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")